# Практикум 1. Неоднозначность, контекст и типология языков

В лаборатории вы видели, как контекст меняет вероятности прочтений слова. Здесь сделаем то же самое на коде:

1. посчитаем энтропию и посмотрим, как она меняется с контекстом;
2. научим наивный байесовский классификатор различать значения слова «ключ» по небольшому корпусу;
3. посмотрим на морфологическую неоднозначность с библиотекой `pymorphy3`;
4. сравним русский, узбекский и английский тексты: длину слов и число форм.

*Практикум к лаборатории 1 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Энтропия: мера неопределённости

У слова «ключ» несколько значений. Каждому значению дадим начальный «счёт» (логит), каждое слово контекста добавляет к счёту свой вес, а softmax превращает счёты в вероятности. Это та же модель, что в лаборатории.

In [ ]:
import numpy as np

def softmax(z):
    z = np.array(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = np.array(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

senses = ["дверной ключ", "родник", "знак в нотах", "ключ к решению"]
prior = [1.0, 0.1, -0.3, 0.5]
cue_weights = {
    "от двери":         [2.8, -1.0, -1.0, -0.5],
    "в лесу":           [0.0,  2.0, -0.8, -0.6],
    "холодный":         [0.2,  1.4, -0.3, -0.4],
    "бил из-под земли": [-2.0, 3.5, -1.0, -1.0],
    "нотный стан":      [-1.0, -1.0, 3.0, -0.5],
    "к задаче":         [-1.0, -1.0, -0.5, 2.8],
}

def show(context):
    z = np.array(prior, dtype=float)
    for c in context:
        z += cue_weights[c]
    p = softmax(z)
    print("контекст:", context or "—")
    for s, x in sorted(zip(senses, p), key=lambda t: -t[1]):
        print(f"   {s:<16} {x:6.1%}")
    print(f"   энтропия = {entropy(p):.2f} бит (максимум {np.log2(len(p)):.2f})\n")

show([])
show(["холодный"])
show(["холодный", "в лесу"])
show(["холодный", "в лесу", "бил из-под земли"])

## 2. Значение слова по контексту: наивный Байес

Настоящие веса никто не задаёт вручную: их учат по корпусу, где у каждого употребления слова отмечено значение. Ниже маленький размеченный корпус: по 6 предложений на значение.

Обратите внимание на морфологию: «ключ», «ключа», «ключом», «двери», «дверь» — разные строки. Сравним два варианта признаков: слова как есть и слова, обрезанные до первых 5 букв (грубая замена лемматизации).

In [ ]:
train = [
    ("Я забыл ключ от квартиры дома", "дверь"),
    ("Ключ от машины лежал на столе", "дверь"),
    ("Он повернул ключ в замке и открыл дверь", "дверь"),
    ("Потерянный ключ нашёлся под ковриком у двери", "дверь"),
    ("Дубликат ключа сделали в мастерской за пять минут", "дверь"),
    ("Ключ застрял в замке старого сарая", "дверь"),
    ("В лесу из-под камня бил холодный ключ", "родник"),
    ("Мы набрали воды из чистого лесного ключа", "родник"),
    ("Горный ключ питает ручей у деревни", "родник"),
    ("Вода в ключе была ледяной даже летом", "родник"),
    ("Туристы остановились у ключа в овраге", "родник"),
    ("Ключ пробился из-под земли у подножия холма", "родник"),
    ("Ключ к решению задачи нашёлся в условии", "решение"),
    ("Учитель дал ключ к пониманию теоремы", "решение"),
    ("Терпение стало ключом к успеху проекта", "решение"),
    ("Ключ к разгадке шифра был в первой строке", "решение"),
    ("Хорошая постановка вопроса ключ к ответу", "решение"),
    ("Этот пример ключ к пониманию всей темы", "решение"),
]
test = [
    ("Холодная вода бьёт из ключа у камня", "родник"),
    ("Ключ от двери висел на гвозде", "дверь"),
    ("Главный ключ к задаче простая идея", "решение"),
    ("Ключ лежал у ручья", "?"),
]

In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

def tokens(text):
    return re.findall(r"[а-яёa-zʻʼ'-]+", text.lower())

def stem5(text):
    # crude stand-in for lemmatization: keep the first 5 letters of every word
    return " ".join(w[:5] for w in tokens(text))

X_text = [s for s, _ in train]
y = [c for _, c in train]

for name, prep in [("слова как есть", lambda s: " ".join(tokens(s))), ("первые 5 букв", stem5)]:
    vec = CountVectorizer(preprocessor=prep, token_pattern=r"[^ ]+")
    X = vec.fit_transform(X_text)
    nb = MultinomialNB(alpha=1.0).fit(X, y)
    print(f"=== {name}: словарь {len(vec.vocabulary_)} признаков")
    for s, gold in test:
        p = nb.predict_proba(vec.transform([s]))[0]
        best = nb.classes_[p.argmax()]
        dist = ", ".join(f"{c} {x:.0%}" for c, x in zip(nb.classes_, p))
        print(f"  {s:<40} → {best:<8} [{dist}]  H={entropy(p):.2f}  (верно: {gold})")
    print()

Какие слова сильнее всего «голосуют» за каждое значение? Посмотрим на логарифмы вероятностей слов в каждом классе (модель с признаками «первые 5 букв»).

In [ ]:
vec = CountVectorizer(preprocessor=stem5, token_pattern=r"[^ ]+")
X = vec.fit_transform(X_text)
nb = MultinomialNB(alpha=1.0).fit(X, y)
vocab = np.array(vec.get_feature_names_out())
for k, c in enumerate(nb.classes_):
    # how much more likely a word is in this class than on average in the others
    other = np.delete(nb.feature_log_prob_, k, axis=0).mean(axis=0)
    score = nb.feature_log_prob_[k] - other
    top = vocab[np.argsort(-score)[:6]]
    print(f"{c:<8}:", ", ".join(top))

## 3. Морфологическая неоднозначность: pymorphy3

`pymorphy3` — морфологический анализатор русского языка на основе словаря OpenCorpora. Для каждой словоформы он возвращает все возможные разборы с оценкой вероятности `score`.

In [ ]:
!pip install -q pymorphy3

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

for word in ["стекло", "мыла", "стали", "печь", "души"]:
    print(word)
    for p in morph.parse(word)[:4]:
        print(f"   {p.normal_form:<10} {str(p.tag):<38} score={p.score:.3f}")

Сколько форм у одного существительного? Метод `lexeme` выдаёт всю парадигму. Обратите внимание: форм получится 14, а не 12. Словарь OpenCorpora учитывает ещё второй родительный и второй предложный падежи («из дому», «на дому»).

In [ ]:
p = morph.parse("дом")[0]
forms = [(f.word, f.tag.case, f.tag.number) for f in p.lexeme]
print(len(forms), "форм:")
for w, case, num in forms:
    print(f"   {w:<8} {case:<5} {num}")

## 4. Типология в цифрах

Один и тот же короткий текст на трёх языках. Узбекский (агглютинативный) выражает грамматику суффиксами, поэтому слов в нём меньше, а сами слова длиннее. Английский (аналитический) использует служебные слова, поэтому слов больше и они короче.

In [ ]:
texts = {
    "ru": [
        "Мы живём в большом доме.",
        "В наших домах тепло и светло.",
        "Дети читают книги в школе.",
        "Учитель дал ученикам новые книги.",
        "Из окна нашего дома видны горы.",
        "Летом мы поедем в деревню к бабушке.",
    ],
    "uz": [
        "Biz katta uyda yashaymiz.",
        "Uylarimizda issiq va yorugʻ.",
        "Bolalar maktabda kitob oʻqiyapti.",
        "Oʻqituvchi oʻquvchilarga yangi kitoblar berdi.",
        "Uyimizning derazasidan togʻlar koʻrinadi.",
        "Yozda biz qishloqqa buvimiznikiga boramiz.",
    ],
    "en": [
        "We live in a big house.",
        "It is warm and bright in our houses.",
        "The children are reading books at school.",
        "The teacher gave the students new books.",
        "The mountains can be seen from the window of our house.",
        "In summer we will go to the village to see our grandmother.",
    ],
}
labels = {"ru": "русский", "uz": "узбекский", "en": "английский"}
import matplotlib.pyplot as plt

stats = {}
for lang, sents in texts.items():
    words = [w for s in sents for w in tokens(s)]
    stats[lang] = (len(words), np.mean([len(w) for w in words]))
    print(f"{labels[lang]:<11} слов: {len(words):3d}   средняя длина слова: {np.mean([len(w) for w in words]):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
names = [labels[k] for k in stats]
ax[0].bar(names, [v[0] for v in stats.values()], color="#2B55C9")
ax[0].set_title("Число слов в тексте")
ax[1].bar(names, [v[1] for v in stats.values()], color="#17907E")
ax[1].set_title("Средняя длина слова, букв")
plt.tight_layout()
plt.show()

## 5. Задания

1. Добавьте в `train` 3–4 предложения про скрипичный ключ (новое значение «ноты») и перезапустите раздел 2. Как изменилась энтропия для предложения «Ключ лежал у ручья»?
2. Сравните предсказания модели «слова как есть» и «первые 5 букв» на тестовых предложениях. Объясните, почему обрезка помогла (или нет), используя понятие словоформы.
3. С помощью `morph.parse` найдите пять слов, у которых три и больше разборов. Какие из них различаются частью речи?
4. Посчитайте долю слов длиннее 8 букв в русском, узбекском и английском текстах из раздела 4. Свяжите результат с типологией.

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь